<a href="https://colab.research.google.com/github/hcy05020-maker/Earth-Engine/blob/GIS/Get_started_with_Earth_Engine_for_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get started with Earth Engine for Python

In [ ]:
#@title Copyright 2024 The Earth Engine Community Authors { display-mode: "form" }
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import ee
import geemap

**2.** Authenticate and initialize the Earth Engine service. Follow the
resulting prompts to complete authentication. Be sure to replace PROJECT_ID
with the name of the project you set up for this quickstart.

In [ ]:
ee.Authenticate()
ee.Initialize(project='my-project-495906')

## What's next

  * Learn about analyzing data with Earth Engine's [objects and methods](https://developers.google.com/earth-engine/guides/objects_methods_overview).
  * Learn about Earth Engine's [processing environments](https://developers.google.com/earth-engine/guides/processing_environments).
  * Learn about Earth Engine's [machine learning capabilities](https://developers.google.com/earth-engine/guides/machine-learning).
  * Learn how to [export your computation results to BigQuery](https://developers.google.com/earth-engine/guides/exporting_to_bigquery).

## 산불 피해지 GIS 분석: 피해 강도 및 경사도

이 섹션에서는 한국의 대형 산불 피해지를 분석하여 산불 피해 강도(dNBR)와 경사도 레이어를 포함하는 GIS를 구축합니다. 2022년 울진-삼척 산불을 예시로 들어 진행합니다.

In [ ]:
# 1. 산불 지역 및 기간 정의 (2022년 울진-삼척 산불 예시)

# 울진-삼척 산불 대략적인 관심 지역(ROI) 정의
# (경도, 위도) 순서로 Point 또는 Polygon을 정의합니다.
fire_roi = ee.Geometry.Polygon([
  [129.1, 36.8],
  [129.5, 36.8],
  [129.5, 37.3], # 북쪽 경계 확장
  [129.1, 37.3], # 북쪽 경계 확장
  [129.1, 36.8]
]);

# 산불 발생 전/후 기간 정의
pre_fire_start = '2022-02-01'
pre_fire_end = '2022-02-28' # 산불 전 (2월)

post_fire_start = '2022-03-01' # 산불 발생 3월 4일경
post_fire_end = '2022-04-30' # 산불 후 (3~4월)

print(f"분석 지역: {fire_roi.getInfo()}")
print(f"산불 전 기간: {pre_fire_start} ~ {pre_fire_end}")
print(f"산불 후 기간: {post_fire_start} ~ {post_fire_end}")

In [ ]:
# 2. 위성 영상 확보 및 전처리

# Sentinel-2 Level-2A (Surface Reflectance) 영상 컬렉션 정의
# DeprecationWarning에 따라 COPERNICUS/S2_SR_HARMONIZED 사용
S2_SR_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'

# Sentinel-2 구름 마스크 함수 정의
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000) # SR 값을 0-1 범위로 스케일링

# NBR(Normalized Burn Ratio) 계산 함수 정의
def calculate_nbr(image):
    # NIR (B8), SWIR2 (B12) 밴드를 사용하여 NBR 계산
    # NBR = (NIR - SWIR2) / (NIR + SWIR2)
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    return image.addBands(nbr)

In [ ]:
# 산불 전 영상 컬렉션 필터링 및 처리
pre_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(pre_fire_start, pre_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 전 영상 선택 (중간값)
pre_fire_image = pre_fire_images.median().clip(fire_roi)

# 산불 후 영상 컬렉션 필터링 및 처리
post_fire_images = (ee.ImageCollection(S2_SR_COLLECTION)
    .filterDate(post_fire_start, post_fire_end)
    .filterBounds(fire_roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # 구름 비율 20% 미만 필터링
    .map(mask_s2_clouds) # 구름 마스크 적용
    .map(calculate_nbr)) # NBR 계산

# 가장 구름 없는 산불 후 영상 선택 (중간값)
post_fire_image = post_fire_images.median().clip(fire_roi)

print("산불 전/후 Sentinel-2 영상 및 NBR 계산 완료.")

In [ ]:
# 3. 산불 피해 강도(dNBR) 계산

# NBR 밴드가 존재하는지 확인 후 계산
if pre_fire_image.bandNames().contains('NBR') and post_fire_image.bandNames().contains('NBR'):
    # dNBR = NBR_pre - NBR_post
    dnbr = pre_fire_image.select('NBR').subtract(post_fire_image.select('NBR')).rename('dNBR')
    print("dNBR 계산 완료.")
else:
    dnbr = ee.Image(0).rename('dNBR') # NBR 밴드가 없으면 0으로 초기화 (오류 방지)
    print("경고: NBR 밴드를 찾을 수 없어 dNBR 계산을 건너뜁니다.")

# 4. 경사도(Slope) 레이어 생성

# SRTM Digital Elevation Model (DEM) 데이터 로드
dem = ee.Image('USGS/SRTMGL1_003').clip(fire_roi)

# 경사도 계산 (도 단위)
slope = ee.Terrain.slope(dem).rename('Slope_Degrees')

print("경사도 레이어 계산 완료.")

In [ ]:
# 5. GIS 레이어

# 지도 객체 생성
m = geemap.Map(center=[37.0, 129.3], zoom=9, height='800px') # 울진-삼척 지역 중심

# dNBR 시각화 파라미터 (일반적인 산불 피해 강도 분류)
# dNBR 값은 보통 -1 ~ 1 사이이며, 양수 값이 클수록 피해가 큼
dnbr_vis_params = {
    'min': -0.5,
    'max': 1.0,
    'palette': [
        '#006400',  # Green (무피해/식생증가)
        '#00FF00',  # Light Green (낮은 피해)
        '#FFFF00',  # Yellow (중간-낮은 피해)
        '#FFA500',  # Orange (중간-높은 피해)
        '#FF0000',  # Red (높은 피해)
        '#8B0000'   # Dark Red (심각한 피해)
    ]
}

# 경사도 시각화 파라미터
slope_vis_params = {
    'min': 0,
    'max': 45, # 최대 45도까지 표시
    'palette': ['lightblue', 'blue', 'darkblue'] # 경사가 높을수록 어두운 파랑
}

# 레이어를 지도에 추가
m.add_layer(pre_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Pre-Fire (RGB)')
m.add_layer(post_fire_image.select(['B4', 'B3', 'B2']), {'min': 0.0, 'max': 0.3}, 'Post-Fire (RGB)')
m.add_layer(dnbr, dnbr_vis_params, 'Burn Severity (dNBR)')
m.add_layer(slope, slope_vis_params, 'Slope')
m.add_colorbar(dnbr_vis_params, label="Burn Severity (dNBR)", orientation='vertical', position='bottomleft')
m.add_colorbar(slope_vis_params, label="Slope (Degrees)", orientation='vertical', position='bottomleft')


## 산사태 위험 지수(Landslide Risk Index, LRI) 계산 및 시각화

산사태 위험 지수는 산불 피해 지역의 경사도(Slope)와 산불 피해 강도(dNBR)를 종합하여 격자 단위로 산사태 발생 가능성을 나타냅니다. 산불로 인해 식생이 소실되면 토양 유실 및 산사태 위험이 증가하므로, dNBR 값이 높고 경사도가 가파른 지역은 더 높은 위험도를 가집니다.

여기서는 다음과 같이 간소화된 위험 수준으로 분류합니다:
- **낮은 위험 (1)**: 경사 < 15도 & dNBR < 0.2
- **중간 위험 (2)**: (15도 <= 경사 < 30도 & dNBR < 0.4) 또는 (경사 < 15도 & 0.2 <= dNBR < 0.4)
- **높은 위험 (3)**: (경사 >= 30도) 또는 (dNBR >= 0.4)
- **매우 높은 위험 (4)**: (경사 >= 40도) & (dNBR >= 0.6)

In [ ]:
# 6. 산사태 위험 지수(LRI) 계산

# LRI 계산 함수 정의
# dNBR과 Slope를 기반으로 산사태 위험도 분류 (예시)
# NBR 값은 -1 ~ 1, Slope는 0 ~ 90 (도) 범위

# dNBR 및 Slope 밴드가 있는지 확인 (없으면 기본값 사용)
dnbr_band = dnbr.select('dNBR')
slope_band = slope.select('Slope_Degrees')

# 초기 위험도 이미지 생성 (기본값 0 또는 낮은 위험)
landslide_risk = ee.Image(0).rename('Landslide_Risk_Index')

# 위험도 분류 (조건문을 사용하여 이미지 픽셀 값 할당)
# 낮은 위험 (1): 경사 < 15 AND dNBR < 0.2
landslide_risk = landslide_risk.where(
    slope_band.lt(15).And(dnbr_band.lt(0.2)),
    ee.Image(1)
)

# 중간 위험 (2): (15 <= 경사 < 30 AND dNBR < 0.4) OR (경사 < 15 AND 0.2 <= dNBR < 0.4)
landslide_risk = landslide_risk.where(
    (slope_band.gte(15).And(slope_band.lt(30)).And(dnbr_band.lt(0.4))).Or(
        slope_band.lt(15).And(dnbr_band.gte(0.2)).And(dnbr_band.lt(0.4))
    ),
    ee.Image(2)
)

# 높은 위험 (3): (경사 >= 30) OR (dNBR >= 0.4) OR (15 <= 경사 < 30 AND 0.4 <= dNBR < 1.0)
landslide_risk = landslide_risk.where(
    (slope_band.gte(30)).Or(dnbr_band.gte(0.4)).Or(
        slope_band.gte(15).And(slope_band.lt(30)).And(dnbr_band.gte(0.4)).And(dnbr_band.lt(1.0))
    ),
    ee.Image(3)
)

# 매우 높은 위험 (4): (경사 >= 40) AND (dNBR >= 0.6)
landslide_risk = landslide_risk.where(
    slope_band.gte(40).And(dnbr_band.gte(0.6)),
    ee.Image(4)
)

print("산사태 위험 지수 계산 완료.")

In [ ]:
# 7. 산사태 위험 지수 시각화

# 산사태 위험 지수 시각화 파라미터 (위험도 1~4)
landslide_risk_vis_params = {
    'min': 1,
    'max': 4,
    'palette': [
        '#00FF00',  # Low Risk (Green)
        '#FFFF00',  # Moderate Risk (Yellow)
        '#FFA500',  # High Risk (Orange)
        '#FF0000'   # Very High Risk (Red)
    ]
}

# dNBR 값을 사용하여 산불 피해 지역만 마스킹
# dNBR이 0.2 이상인 지역만 표시 (산불 피해가 있다고 간주)
burn_mask = dnbr.select('dNBR').gt(0.2) # dNBR 밴드가 0.2보다 큰 지역에만 마스크 적용
landslide_risk_masked = landslide_risk.updateMask(burn_mask)

# 레이어를 지도에 추가
m.add_layer(landslide_risk_masked, landslide_risk_vis_params, 'Landslide Risk Index (Burned Areas)')
m.add_colorbar(landslide_risk_vis_params,
               label='Landslide Risk Index (1=Low, 4=Very High)',
               orientation='vertical',
               position='bottomleft')



In [ ]:
import numpy as np
import ee

# Calculate approximate degree equivalent for 1km at the region's latitude
# At ~37 degrees latitude, 1 degree latitude is ~111 km, 1 degree longitude is ~89 km.
# So, 1km is roughly 0.009 degrees latitude and 0.011 degrees longitude.
lat_interval_degrees = 0.009
lon_interval_degrees = 0.011

# Always use fire_roi bounds for the grid generation
# Extract min/max lat/lon from fire_roi
fire_coords = fire_roi.getInfo()['coordinates'][0]
west = min(c[0] for c in fire_coords)
south = min(c[1] for c in fire_coords)
east = max(c[0] for c in fire_coords)
north = max(c[1] for c in fire_coords)

grid_features = []

# Generate latitude lines
# Add a small buffer to ensure lines cover the visible area completely
for lat in np.arange(south - lat_interval_degrees, north + lat_interval_degrees, lat_interval_degrees):
    line = ee.Geometry.LineString([[west - lon_interval_degrees, lat], [east + lon_interval_degrees, lat]])
    grid_features.append(ee.Feature(line))

# Generate longitude lines
for lon in np.arange(west - lon_interval_degrees, east + lat_interval_degrees, lon_interval_degrees):
    line = ee.Geometry.LineString([[lon, south - lat_interval_degrees], [lon, north + lat_interval_degrees]])
    grid_features.append(ee.Feature(line))

grid_collection = ee.FeatureCollection(grid_features)

# Add the manually created grid as a layer
m.add_layer(grid_collection, {'color': 'gray'}, '1km Grid')


In [ ]:
# 1. fire_roi 내부에 1km 격자 폴리곤 생성
# fire_roi의 경계를 기반으로 1km 간격의 그리드 폴리곤 생성
fire_coords_1km = fire_roi.getInfo()['coordinates'][0]
min_lon_roi_1km = min(c[0] for c in fire_coords_1km)
min_lat_roi_1km = min(c[1] for c in fire_coords_1km)
max_lon_roi_1km = max(c[0] for c in fire_coords_1km)
max_lat_roi_1km = max(c[1] for c in fire_coords_1km)

# lat_interval_degrees, lon_interval_degrees는 cell 672fb44a에서 1km에 해당하는 값으로 정의됨

grid_polygons_1km_list = []
current_lat_gen = min_lat_roi_1km
while current_lat_gen < max_lat_roi_1km + 1e-9: # 작은 오차를 고려하여 경계까지 포함
    current_lon_gen = min_lon_roi_1km
    while current_lon_gen < max_lon_roi_1km + 1e-9: # 작은 오차를 고려하여 경계까지 포함
        square_coords_1km = [
            [current_lon_gen, current_lat_gen],
            [current_lon_gen + lon_interval_degrees, current_lat_gen],
            [current_lon_gen + lon_interval_degrees, current_lat_gen + lat_interval_degrees],
            [current_lon_gen, current_lat_gen + lat_interval_degrees],
            [current_lon_gen, current_lat_gen]
        ]
        polygon_1km = ee.Geometry.Polygon(square_coords_1km)
        grid_polygons_1km_list.append(ee.Feature(polygon_1km))
        current_lon_gen += lon_interval_degrees
    current_lat_gen += lat_interval_degrees

# 1km 격자 폴리곤들의 FeatureCollection 생성
grid_1km_polygons_fc = ee.FeatureCollection(grid_polygons_1km_list)

print(f"생성된 1km 격자 폴리곤 수: {grid_1km_polygons_fc.size().getInfo()}")

In [ ]:
# 2. 1km 격자 폴리곤 전체 영역의 바운딩 박스 계산
# 이 바운딩 박스를 4x4로 나눌 기준 영역으로 사용합니다.
grid_1km_total_bounds_info = grid_1km_polygons_fc.geometry().bounds().getInfo()['coordinates'][0]
grid_min_lon = grid_1km_total_bounds_info[0][0]
grid_min_lat = grid_1km_total_bounds_info[0][1]
grid_max_lon = grid_1km_total_bounds_info[2][0]
grid_max_lat = grid_1km_total_bounds_info[2][1]

# 3. 바운딩 박스를 4x4로 나누기 위한 경위도 간격 계산
lon_range = grid_max_lon - grid_min_lon
lat_range = grid_max_lat - grid_min_lat

lon_step_4x4 = lon_range / 4.0
lat_step_4x4 = lat_range / 4.0

# 4. 중앙 4개 칸(2x2)의 경계 정의 (0-indexed 4x4 그리드의 row 1-2, col 1-2)
central_min_lat = grid_min_lat + lat_step_4x4
central_max_lat = grid_min_lat + 3 * lat_step_4x4
central_min_lon = grid_min_lon + lon_step_4x4
central_max_lon = grid_min_lon + 3 * lon_step_4x4

# 중앙 4개 칸을 나타내는 지오메트리 생성
central_4_cells_boundary = ee.Geometry.Polygon([
    [[central_min_lon, central_min_lat],
     [central_max_lon, central_min_lat],
     [central_max_lon, central_max_lat],
     [central_min_lon, central_max_lat],
     [central_min_lon, central_min_lat]]
])

# 5. 중앙 4개 칸에 포함되는 1km 격자 폴리곤 필터링
central_1km_grids_fc = grid_1km_polygons_fc.filterBounds(central_4_cells_boundary)

print(f"중앙 4개 칸에 포함된 1km 격자 폴리곤 수: {central_1km_grids_fc.size().getInfo()}")

In [ ]:
# 7. 선택된 10개 1km 격자 및 중앙 영역 시각화

# 중앙 4개 칸의 경계를 지도에 추가
m.add_layer(central_4_cells_boundary, {'color': 'orange', 'fillColor': '00000000', 'width': 3}, 'Central 4x4 Grid Boundary')



## 새로운 제한 영역 정의 (새로운 4x4 그리드의 중앙 2x2 영역)

In [ ]:
central_4_cells_total_bounds_info = central_4_cells_boundary.getInfo()['coordinates'][0]
central_grid_min_lon = central_4_cells_total_bounds_info[0][0]
central_grid_min_lat = central_4_cells_total_bounds_info[0][1]
central_grid_max_lon = central_4_cells_total_bounds_info[2][0]
central_grid_max_lat = central_4_cells_total_bounds_info[2][1]

central_lon_range = central_grid_max_lon - central_grid_min_lon
central_lat_range = central_grid_max_lat - central_grid_min_lat

# 중앙 영역을 4x4로 나눈 것의 중앙 2x2 영역을 가정 (조금 더 제한된 영역)
new_restricted_min_lat = central_grid_min_lat + (central_lat_range / 4.0)
new_restricted_max_lat = central_grid_min_lat + (3 * central_lat_range / 4.0)
new_restricted_min_lon = central_grid_min_lon + (central_lon_range / 4.0)
new_restricted_max_lon = central_grid_min_lon + (3 * central_lon_range / 4.0)

new_restricted_boundary = ee.Geometry.Polygon([
    [new_restricted_min_lon, new_restricted_min_lat],
    [new_restricted_max_lon, new_restricted_min_lat],
    [new_restricted_max_lon, new_restricted_max_lat],
    [new_restricted_min_lon, new_restricted_max_lat],
    [new_restricted_min_lon, new_restricted_min_lat]
])

print(f"새롭게 제한된 경계 영역: {new_restricted_boundary.getInfo()}")

# 새로운 제한 영역의 경계를 지도에 추가
m.add_layer(new_restricted_boundary, {'color': 'blue', 'fillColor': '0000FF40', 'width': 3}, 'New Restricted Grid Boundary')

In [ ]:
# 3. 기존 중앙 1km 격자들 중에서 새로운 제한 영역에 포함되는 격자 필터링
filtered_central_1km_grids_fc = central_1km_grids_fc.filterBounds(new_restricted_boundary)

print(f"새로운 제한 영역에 포함된 1km 격자 폴리곤 수: {filtered_central_1km_grids_fc.size().getInfo()}")

In [ ]:
# 새로운 제한 영역의 경계를 지도에 추가
m.add_layer(new_restricted_boundary, {'color': 'blue', 'fillColor': '0000FF40', 'width': 3}, 'New Restricted Grid Boundary')

m # 최종 지도 표시

## 사용자 지정 1km 격자 선택 및 시각화

이전 단계에서 정의된 제한된 영역 내에서 사용자가 지정한 상대적 위치를 기반으로 3개의 1km 격자를 선택하고 지도에 표시합니다.

In [ ]:
# 1. 제한된 영역의 총 행/열 수 계산
# Python float 값을 ee.Number로 변환하여 계산의 정확성을 높입니다。
num_rows_in_restricted_area = ee.Number((ee.Number(new_restricted_max_lat).subtract(ee.Number(new_restricted_min_lat))).divide(ee.Number(lat_interval_degrees))).round()
num_cols_in_restricted_area = ee.Number((ee.Number(new_restricted_max_lon).subtract(ee.Number(new_restricted_min_lon))).divide(ee.Number(lon_interval_degrees))).round()

# 계산된 행/열 수를 클라이언트 측에서 사용하기 위해 getInfo() 호출
num_rows_val = num_rows_in_restricted_area.getInfo()
num_cols_val = num_cols_in_restricted_area.getInfo()

print(f"제한된 영역은 대략 {num_rows_val}행과 {num_cols_val}열을 가집니다.")

# 2. 사용자 요청에 따른 0-인덱스 격자 위치 계산 (하단-좌측을 (0,0)으로 가정)

# 1번: 오른쪽에서 5번째 (col), 위에서 3번째 (row) -> 아래로 3칸
target1_col_idx_from_left = num_cols_val - 5
target1_row_idx_from_bottom = num_rows_val - 3 - 3 # Adjusted: move 3 cells down

# 2번: 오른쪽 맨 끝 (col), 아래에서 5번째 (row) -> 위로 1칸
target2_col_idx_from_left = num_cols_val - 1
target2_row_idx_from_bottom = 4 + 1 # Adjusted: move 1 cell up (0-indexed from bottom)

# 3번: 왼쪽에서 6번째 (col), 위에서 10번째 (row) -> 아래로 3칸
target3_col_idx_from_left = 5 # 0-인덱스에서 6번째는 5
target3_row_idx_from_bottom = num_rows_val - 10 - 3 # Adjusted: move 3 cells down

# 20m 위로 올리기 위한 위도 오프셋 계산 (1km = lat_interval_degrees)
offset_20m_lat = ee.Number(lat_interval_degrees).multiply(0.02) # 1km의 2% = 20m

# 3. 특정 격자 위치에 해당하는 ee.Geometry.Polygon 생성 헬퍼 함수
def create_grid_polygon_at_index(col_idx, row_idx, min_lon_origin, min_lat_origin, lon_interval, lat_interval, vertical_offset=0):
    cell_min_lon = ee.Number(min_lon_origin).add(ee.Number(col_idx).multiply(ee.Number(lon_interval)))
    cell_min_lat = ee.Number(min_lat_origin).add(ee.Number(row_idx).multiply(ee.Number(lat_interval))).add(ee.Number(vertical_offset))
    cell_max_lon = cell_min_lon.add(ee.Number(lon_interval))
    cell_max_lat = cell_min_lat.add(ee.Number(lat_interval))
    return ee.Geometry.Polygon(
        [
            [[
                cell_min_lon,
                cell_min_lat
            ], [
                cell_max_lon,
                cell_min_lat
            ], [
                cell_max_lon,
                cell_max_lat
            ], [
                cell_min_lon,
                cell_max_lat
            ], [
                cell_min_lon,
                cell_min_lat
            ]]
        ]
    )

# 제한된 영역의 시작 경위도 및 격자 간격
# Python float 값을 ee.Number로 변환하여 EE API와 호환되게 합니다.
ee_restricted_min_lon = ee.Number(new_restricted_min_lon)
ee_restricted_min_lat = ee.Number(new_restricted_min_lat)
ee_lon_interval_degrees = ee.Number(lon_interval_degrees)
ee_lat_interval_degrees = ee.Number(lat_interval_degrees)

# 4. 세 개의 목표 격자 지오메트리 생성
target_geometries_list = [
    create_grid_polygon_at_index(target1_col_idx_from_left, target1_row_idx_from_bottom,
                                 ee_restricted_min_lon, ee_restricted_min_lat,
                                 ee_lon_interval_degrees, ee_lat_interval_degrees, vertical_offset=offset_20m_lat),
    create_grid_polygon_at_index(target2_col_idx_from_left, target2_row_idx_from_bottom,
                                 ee_restricted_min_lon, ee_restricted_min_lat,
                                 ee_lon_interval_degrees, ee_lat_interval_degrees, vertical_offset=offset_20m_lat),
    create_grid_polygon_at_index(target3_col_idx_from_left, target3_row_idx_from_bottom,
                                 ee_restricted_min_lon, ee_restricted_min_lat,
                                 ee_lon_interval_degrees, ee_lat_interval_degrees, vertical_offset=offset_20m_lat)
]

# 5. 세 개의 목표 격자 지오메트리를 Feature로 변환하여 FeatureCollection 생성
selected_grid_features = []
for i, geom in enumerate(target_geometries_list):
    selected_grid_features.append(ee.Feature(geom, {'name': f'User Selected Grid {i+1}'}))

selected_grid_collection = ee.FeatureCollection(selected_grid_features)

print(f"사용자 선택 1km 격자 수: {selected_grid_collection.size().getInfo()}")

# 6. 선택된 격자들을 지도에 추가 (fillColor 제거, width 통일)
m.add_layer(selected_grid_collection, {'color': 'red', 'fillColor': '00000000', 'width': 1}, 'User Selected 1km Grids')

# 7. 선택된 격자들을 중심으로 지도 이동
m.centerObject(selected_grid_collection, zoom=12)

m

## 선택된 1km 격자에 100m 격자 세분화 및 시각화

이 섹션에서는 이전에 선택된 3개의 1km 격자 각각에 대해 100m 간격의 세분화된 격자 폴리곤을 생성하고 지도에 추가합니다. 이는 특정 관심 영역 내에서 더 상세한 공간 분석을 위한 기반을 마련합니다.

In [ ]:
# 1. 100m 격자 생성을 위한 경위도 간격 정의 (전역 변수는 이제 참고용)
# 위도 약 37도에서 100m에 해당하는 대략적인 경위도 값 (1km 값의 1/10)
lat_interval_degrees_100m_approx = lat_interval_degrees / 10.0 # 1km_lat_interval / 10
lon_interval_degrees_100m_approx = lon_interval_degrees / 10.0 # 1km_lon_interval / 10

all_100m_subgrids = []

# 2. 각 선택된 1km 격자에 대해 100m 하위 격자 생성
for idx, feature in enumerate(selected_grid_collection.getInfo()['features']):
    geom_1km = ee.Feature(feature).geometry()
    bounds_1km = geom_1km.bounds().getInfo()['coordinates'][0]

    min_lon_1km = bounds_1km[0][0]
    min_lat_1km = bounds_1km[0][1]
    max_lon_1km = bounds_1km[2][0]
    max_lat_1km = bounds_1km[2][1]

    # 각 1km 격자의 실제 경계에 맞춰 100m 스텝 계산
    actual_width_deg = max_lon_1km - min_lon_1km
    actual_height_deg = max_lat_1km - min_lat_1km

    lon_step_for_100m = actual_width_deg / 10.0
    lat_step_for_100m = actual_height_deg / 10.0

    # 사용자의 요청에 따라 20m 위로, 130m 오른쪽으로 이동합니다.
    # 100m에 해당하는 lat/lon 스텝을 기준으로 오프셋 계산
    vertical_offset_degrees = 0.2 * lat_step_for_100m # 20m 위로 이동 (양수 위도)
    horizontal_offset_degrees = 1.3 * lon_step_for_100m # 130m 오른쪽으로 이동 (양수 경도)

    # 1km 격자 내부에 정확히 10x10개의 100m 하위 격자를 생성합니다.
    for row in range(10): # 위도 방향 (남쪽에서 북쪽으로)
        current_lat_100m_start = min_lat_1km + row * lat_step_for_100m + vertical_offset_degrees
        current_lat_100m_end = min_lat_1km + (row + 1) * lat_step_for_100m + vertical_offset_degrees
        for col in range(10): # 경도 방향 (서쪽에서 동쪽으로)
            current_lon_100m_start = min_lon_1km + col * lon_step_for_100m + horizontal_offset_degrees
            current_lon_100m_end = min_lon_1km + (col + 1) * lon_step_for_100m + horizontal_offset_degrees

            square_coords_100m = [
                [current_lon_100m_start, current_lat_100m_start],
                [current_lon_100m_end, current_lat_100m_start],
                [current_lon_100m_end, current_lat_100m_end],
                [current_lon_100m_start, current_lat_100m_end],
                [current_lon_100m_start, current_lat_100m_start]
            ]
            polygon_100m = ee.Geometry.Polygon(square_coords_100m)
            all_100m_subgrids.append(ee.Feature(polygon_100m))

# 3. 모든 100m 격자 폴리곤을 FeatureCollection으로 생성
all_100m_subgrids_fc = ee.FeatureCollection(all_100m_subgrids)

print(f"총 {all_100m_subgrids_fc.size().getInfo()}개의 100m 격자 폴리곤이 생성되었습니다.")

# 4. 100m 격자 레이어를 지도에 추가 (채우기 색상 제거)
#m.add_layer(all_100m_subgrids_fc, {'color': 'green', 'fillColor': '00000000','width': 0.5}, '100m Subgrids in Selected 1km Grids')

# 피처 컬렉션에 직접 스타일을 입힌 뒤 맵에 추가
styled_fc = all_100m_subgrids_fc.style(
    color='black',
    fillColor='00000000',
    width=0.4
)

m.add_layer(styled_fc, {}, '100m Subgrids in Selected 1km Grids')
m

### 선택된 첫 번째 1km 격자의 10m 하위 격자별 GIS 정보 추출 및 CSV 저장

### Load the exported GIS data and analyze dNBR values

In [ ]:
import pandas as pd

# Assuming the user has uploaded the CSV file named 'grid_1_10m_subgrid_gis_data.csv'
# to the Colab environment's /content/sample_data directory.
try:
    df_gis_data = pd.read_csv('/content/sample_data/grid_1_10m_subgrid_gis_data.csv')
    print("CSV file loaded successfully!")
    display(df_gis_data.head())
except FileNotFoundError:
    print("Error: 'grid_1_10m_subgrid_gis_data.csv' not found. Please ensure the file is uploaded to Colab.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

In [ ]:
# Filter data for dNBR values between 0.27 and 0.65
if 'df_gis_data' in locals() and not df_gis_data.empty:
    filtered_dnbr_data = df_gis_data[(df_gis_data['dnbr'] >= 0.27) & (df_gis_data['dnbr'] <= 0.65)]
    count = len(filtered_dnbr_data)
    print(f"Number of data points with dNBR between 0.27 and 0.65: {count}")
else:
    print("DataFrame 'df_gis_data' is not available or is empty. Please ensure the CSV is loaded correctly.")

### Land Cover Classification Mapping

Each numerical value in the `land_cover` column corresponds to a specific land cover type, according to the ESA WorldCover 10m v100 classification scheme:

*   **0**: Tree cover
*   **10**: Shrubland
*   **20**: Grassland
*   **30**: Cropland
*   **40**: Built-up
*   **50**: Bare / sparse vegetation
*   **60**: Snow and ice
*   **70**: Permanent water bodies
*   **80**: Herbaceous wetland
*   **90**: Mangroves
*   **100**: Moss and lichen

### Visualize Selected dNBR Pixels on Map

In [ ]:
import ee
import pandas as pd

# Convert the filtered pandas DataFrame to an Earth Engine FeatureCollection
def df_to_feature_collection(df):
    features = []
    # Columns to include as properties in the Earth Engine Feature
    # Exclude 'longitude', 'latitude', and '.geo' as they define the geometry or are internal EE fields
    # 'system:index' can also be problematic if not handled as a string or explicitly excluded based on context.
    # We'll explicitly handle 'system:index' as a string.
    property_cols_to_include = [
        col for col in df.columns
        if col not in ['longitude', 'latitude', '.geo'] and col != 'system:index'
    ]

    for index, row in df.iterrows():
        # Create a point geometry from latitude and longitude
        geom = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])

        # Create a dictionary of properties from the selected columns
        properties = {}
        for col in property_cols_to_include:
            # Convert numpy types to native Python types
            properties[col] = row[col].item() if hasattr(row[col], 'item') else row[col]

        # Explicitly add 'system:index' as a string if it exists
        if 'system:index' in row.index:
            properties['system:index'] = str(row['system:index'])

        feature = ee.Feature(geom, properties)
        features.append(feature)
    return ee.FeatureCollection(features)

# Create a FeatureCollection for the filtered dNBR data
# Assuming filtered_dnbr_data is available in the kernel from previous steps
selected_dnbr_fc = df_to_feature_collection(filtered_dnbr_data)

print(f"Created Earth Engine FeatureCollection with {selected_dnbr_fc.size().getInfo()} features for selected dNBR pixels.")

In [ ]:
# Define visualization parameters for the selected pixels
selected_pixels_vis_params = {
    'color': 'magenta',
    'pointSize': 3,
    'width': 1 # For point geometries, width is usually not directly applicable, but can be used for polygon outlines if features are polygons
}

# Add the selected pixels to the map
m.add_layer(selected_dnbr_fc, selected_pixels_vis_params, 'Selected dNBR Pixels (0.27-0.65)')

# Center the map on the first selected pixel, or the overall ROI
m.centerObject(selected_dnbr_fc, zoom=12)

m

### Visualize dNBR Pixels (0.44-0.66) on Map

In [ ]:
# Filter data for dNBR values between 0.44 and 0.66
if 'df_gis_data' in locals() and not df_gis_data.empty:
    filtered_dnbr_data_new_range = df_gis_data[(df_gis_data['dnbr'] >= 0.44) & (df_gis_data['dnbr'] <= 0.66)]
    count_new_range = len(filtered_dnbr_data_new_range)
    print(f"Number of data points with dNBR between 0.44 and 0.66: {count_new_range}")
else:
    print("DataFrame 'df_gis_data' is not available or is empty. Please ensure the CSV is loaded correctly.")

# Create a FeatureCollection for the newly filtered dNBR data
selected_dnbr_fc_new_range = df_to_feature_collection(filtered_dnbr_data_new_range)

print(f"Created Earth Engine FeatureCollection with {selected_dnbr_fc_new_range.size().getInfo()} features for dNBR pixels (0.44-0.66).")

In [ ]:
# Define visualization parameters for the newly selected pixels
new_selected_pixels_vis_params = {
    'color': 'skyblue', # Using a different color for distinction
    'pointSize': 3,
    'width': 1
}

# Add the newly selected pixels to the map
m.add_layer(selected_dnbr_fc_new_range, new_selected_pixels_vis_params, 'Selected dNBR Pixels (0.44-0.66)')

# Center the map on the new selected pixels
m.centerObject(selected_dnbr_fc_new_range, zoom=12)

m